In [1]:
# =============================================================================
# MODÈLE FINAL - LightGBM pour Prédiction de Réachat (Instacart)
# Version complète, claire et prête à tester - Projet DSTI
# =============================================================================

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [3]:
print("=== MODÈLE LIGHTGBM - PRÉDICTION DE RÉACHAT ===")

=== MODÈLE LIGHTGBM - PRÉDICTION DE RÉACHAT ===


In [49]:
# -----------------------------------------------------------------------------
# CONFIGURATION – À ADAPTER
# -----------------------------------------------------------------------------
DATA_PATH = "../Data/"           # ← Mets le chemin où sont tes fichiers CSV
SAMPLE_USERS = 50000                 # Augmente si ta machine le permet (max ~200k)

print("=== PIPELINE COMPLET – Début ===")

=== PIPELINE COMPLET – Début ===


In [4]:
DATA_PATH = '../Data/' 
# -----------------------------------------------------------------------------
# 2. Chargement des fichiers
# -----------------------------------------------------------------------------
print("\nChargement des fichiers Instacart...")

orders = pd.read_csv(DATA_PATH + 'orders.csv')
prior = pd.read_csv(DATA_PATH + 'order_products__prior.csv')
train = pd.read_csv(DATA_PATH + 'order_products__train.csv')
products = pd.read_csv(DATA_PATH + 'products.csv')

print(f"orders     : {orders.shape}")
print(f"prior      : {prior.shape}")
print(f"train      : {train.shape}")
print(f"products   : {products.shape}")


Chargement des fichiers Instacart...
orders     : (3421083, 7)
prior      : (32434489, 4)
train      : (1384617, 4)
products   : (49688, 4)


In [50]:
# -----------------------------------------------------------------------------
# 2. Échantillonnage (pour que ça tourne vite)
# -----------------------------------------------------------------------------
sampled_users = orders['user_id'].sample(n=SAMPLE_USERS, random_state=42).unique()
orders_sample = orders[orders['user_id'].isin(sampled_users)].copy()
prior_sample = prior[prior['order_id'].isin(orders_sample['order_id'])].copy()
train_sample = train[train['order_id'].isin(orders_sample['order_id'])].copy()

In [51]:
# -----------------------------------------------------------------------------
# 3. Création de prior_with_user (merge indispensable)
# -----------------------------------------------------------------------------
prior_with_user = prior_sample.merge(
    orders_sample[['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']],
    on='order_id',
    how='left'
)

print("Colonnes dans prior_with_user :", prior_with_user.columns.tolist())

Colonnes dans prior_with_user : ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']


In [52]:
# -----------------------------------------------------------------------------
# 4. Création de df_pred (paires user-product + label)
# -----------------------------------------------------------------------------
user_products = prior_with_user.groupby('user_id')['product_id'].unique().reset_index()
user_products = user_products.explode('product_id')

labels = train_sample[['order_id', 'product_id']].copy()
labels['label'] = 1
labels = labels.merge(orders_sample[['order_id', 'user_id']], on='order_id', how='left')

df_pred = user_products.merge(labels[['user_id', 'product_id', 'label']], 
                              on=['user_id', 'product_id'], how='left')
df_pred['label'] = df_pred['label'].fillna(0).astype(int)

print(f"df_pred créé : {df_pred.shape}")

df_pred créé : (3874966, 3)


In [53]:
# -----------------------------------------------------------------------------
# 5. Feature Engineering minimal mais efficace
# -----------------------------------------------------------------------------
print("Feature Engineering...")

# User features
user_feats = orders_sample.groupby('user_id').agg(
    user_total_orders=('order_number', 'max'),
    user_avg_days_since_prior=('days_since_prior_order', 'mean')
).reset_index()

user_reorder = prior_with_user.groupby('user_id')['reordered'].mean().reset_index(name='user_reorder_rate')

user_features = user_feats.merge(user_reorder, on='user_id', how='left')

# Product features
product_features = prior_with_user.groupby('product_id').agg(
    product_total_purchases=('order_id', 'count'),
    product_reorder_rate=('reordered', 'mean')
).reset_index()

# User-product features
up_features = prior_with_user.groupby(['user_id', 'product_id']).agg(
    user_product_orders=('order_id', 'nunique'),
    user_product_last_order=('order_number', 'max'),
    user_product_reorder_rate=('reordered', 'mean')
).reset_index()

# Merge tout
df = df_pred.merge(user_features, on='user_id', how='left')
df = df.merge(product_features, on='product_id', how='left')
df = df.merge(up_features, on=['user_id', 'product_id'], how='left')

# Features dérivées
df['user_product_order_ratio'] = df['user_product_orders'] / df['user_total_orders'].replace(0, 1)
df['days_since_last'] = df['user_total_orders'] - df['user_product_last_order']


# Prix simulé
np.random.seed(42)
products['price'] = np.random.uniform(0.5, 15, len(products))
df = df.merge(products[['product_id', 'price']], on='product_id', how='left')

df = df.fillna(0)  # simple pour ce test

print(f"df final prêt : {df.shape} lignes × {df.shape[1]} colonnes")

Feature Engineering...
df final prêt : (3874966, 14) lignes × 14 colonnes


In [54]:
# -----------------------------------------------------------------------------
# 6. Modèle LightGBM
# -----------------------------------------------------------------------------
exclude = ['user_id', 'product_id', 'label']
features = [c for c in df.columns if c not in exclude]

X = df[features]
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'verbose': -1
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(100)]
)

Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.15379


In [55]:
# Prédictions
val_proba = model.predict(X_val)
pred = (val_proba > 0.25).astype(int)  # seuil simple

In [56]:
# Métriques
print("\nRésultats :")
print(f"F1-score : {f1_score(y_val, pred):.4f}")
print(f"Precision : {precision_score(y_val, pred):.4f}")
print(f"Recall : {recall_score(y_val, pred):.4f}")
print(f"ROC AUC : {roc_auc_score(y_val, val_proba):.4f}")


Résultats :
F1-score : 0.3176
Precision : 0.3775
Recall : 0.2741
ROC AUC : 0.8453


In [ ]:
# Sauvegarde
model.save_model('model_reorder.txt')
print("Modèle sauvegardé : model_reorder.txt")

In [48]:
# =============================================================================
# TEST RÉEL : Prédictions concrètes pour un utilisateur (comme dans une vraie app)
# =============================================================================

print("\n=== TEST RÉEL DES PRÉDICTIONS ===")

# 1. Choisir un utilisateur qui existe dans ton df et qui a des données intéressantes
# (on prend un utilisateur qui a déjà acheté plusieurs produits pour que ce soit utile)
users_with_products = df['user_id'].value_counts()
user_example = users_with_products[users_with_products >= 10].index[0]  # au moins 10 produits déjà achetés

print(f"Utilisateur test choisi : user_id = {user_example}")
print(f"Ce client a déjà acheté {users_with_products[user_example]} produits différents")

# 2. Extraire toutes les lignes de cet utilisateur (tous les produits qu'il a déjà achetés)
user_data = df[df['user_id'] == user_example].copy()

# 3. Prédire les probabilités de réachat pour tous ses produits
user_data['proba_reachat'] = model.predict(user_data[features])

# 4. Ajouter le nom du produit (pour que ce soit lisible)
user_data = user_data.merge(
    products[['product_id', 'product_name']],
    on='product_id',
    how='left'
)

# 5. Trier par probabilité descendante et prendre le top 10
top_recommendations = user_data.sort_values('proba_reachat', ascending=False).head(10)

# 6. Affichage clair et joli
print(f"\nTop 10 produits les plus susceptibles d'être ré-achetés par user_{user_example} :")
display_df = top_recommendations[[
    'product_name', 
    'proba_reachat', 
    'price', 
    'user_product_orders', 
    'label'   # 1 = a vraiment été ré-acheté (dans le set train)
]].copy()

display_df = display_df.round({'proba_reachat': 4, 'price': 2})
display_df.columns = ['Produit', 'Probabilité de réachat', 'Prix simulé (€)', 'Nb commandes précédent', 'Vraiment ré-acheté ?']

print(display_df.to_string(index=False))

# 7. Résumé rapide
nb_vrais_reachats_dans_top10 = (display_df['Vraiment ré-acheté ?'] == 1).sum()
print(f"\nRésultat du test : le modèle a prédit {nb_vrais_reachats_dans_top10}/10 produits qui ont vraiment été ré-achetés")
print(f"Gain potentiel estimé (si marge 30 %) : {display_df['Prix simulé (€)'].sum() * 0.3:.2f} € par commande")


=== TEST RÉEL DES PRÉDICTIONS ===
Utilisateur test choisi : user_id = 182401
Ce client a déjà acheté 726 produits différents

Top 10 produits les plus susceptibles d'être ré-achetés par user_182401 :
                  Produit  Probabilité de réachat  Prix simulé (€)  Nb commandes précédent  Vraiment ré-acheté ?
   Bag of Organic Bananas                  0.3210            11.15                      53                     0
     Organic Baby Spinach                  0.2998            12.97                      40                     1
     Pure Sparkling Water                  0.2907             1.21                      53                     0
         Organic Zucchini                  0.2455             8.24                      34                     1
 Sea Salt Roasted Seaweed                  0.2112             6.33                      28                     0
             Hass Avocado                  0.1917             8.15                      13                     0
 Nonfat 

In [70]:
# =============================================================================
# TEST RÉEL : Prédictions concrètes pour un utilisateur (comme dans une vraie app)
# =============================================================================

print("\n=== TEST RÉEL DES PRÉDICTIONS ===")

# 1. Choisir un utilisateur qui existe dans ton df et qui a des données intéressantes
# (on prend un utilisateur qui a déjà acheté plusieurs produits pour que ce soit utile)
users_with_products = df['user_id'].value_counts()
user_example = 206201 # au moins 10 produits déjà achetés

print(f"Utilisateur test choisi : user_id = {user_example}")
print(f"Ce client a déjà acheté {users_with_products[user_example]} produits différents")

# 2. Extraire toutes les lignes de cet utilisateur (tous les produits qu'il a déjà achetés)
user_data = df[df['user_id'] == user_example].copy()

# 3. Prédire les probabilités de réachat pour tous ses produits
user_data['proba_reachat'] = model.predict(user_data[features])

# 4. Ajouter le nom du produit (pour que ce soit lisible)
user_data = user_data.merge(
    products[['product_id', 'product_name']],
    on='product_id',
    how='left'
)

# 5. Trier par probabilité descendante et prendre le top 10
top_recommendations = user_data.sort_values('proba_reachat', ascending=False).head(10)

# 6. Affichage clair et joli
print(f"\nTop 10 produits les plus susceptibles d'être ré-achetés par user_{user_example} :")
display_df = top_recommendations[[
    'product_name', 
    'proba_reachat', 
    'price', 
    'user_product_orders', 
    'label'   # 1 = a vraiment été ré-acheté (dans le set train)
]].copy()

display_df = display_df.round({'proba_reachat': 4, 'price': 2})
display_df.columns = ['Produit', 'Probabilité de réachat', 'Prix simulé (€)', 'Nb commandes précédent', 'Vraiment ré-acheté ?']

print(display_df.to_string(index=False))

# 7. Résumé rapide
nb_vrais_reachats_dans_top10 = (display_df['Vraiment ré-acheté ?'] == 1).sum()
print(f"\nRésultat du test : le modèle a prédit {nb_vrais_reachats_dans_top10}/10 produits qui ont vraiment été ré-achetés")
print(f"Gain potentiel estimé (si marge 30 %) : {display_df['Prix simulé (€)'].sum() * 0.3:.2f} € par commande")


=== TEST RÉEL DES PRÉDICTIONS ===
Utilisateur test choisi : user_id = 206201
Ce client a déjà acheté 174 produits différents

Top 10 produits les plus susceptibles d'être ré-achetés par user_206201 :
                                       Produit  Probabilité de réachat  Prix simulé (€)  Nb commandes précédent  Vraiment ré-acheté ?
                           2% Reduced Fat Milk                  0.3417             8.84                      24                     0
                               Sourdough Bread                  0.2965             1.77                      21                     0
                                  Classic Soda                  0.2297             4.02                      13                     0
                                  Energy Drink                  0.2272            12.78                      11                     0
Heavy Whipping Crean Ultra Pasteurized Grade A                  0.1659             3.59                       7                  

In [69]:
# Test interactif : change le numéro ci-dessous
user_id_a_tester = 	206201   # ← Mets n'importe quel user_id qui existe dans df

if user_id_a_tester in df['user_id'].values:
    user_data = df[df['user_id'] == user_id_a_tester].copy()
    user_data['proba_reachat'] = model.predict(user_data[features])
    user_data = user_data.merge(products[['product_id', 'product_name']], on='product_id')
    
    top_10 = user_data.sort_values('proba_reachat', ascending=False).head(10)
    print(f"Recommandations pour user_id {user_id_a_tester} :")
    print(top_10[['product_name', 'proba_reachat', 'price', 'label']])
else:
    print(f"User {user_id_a_tester} n'existe pas dans l'échantillon d'entraînement.")

Recommandations pour user_id 206201 :
                                      product_name  proba_reachat      price  \
24                             2% Reduced Fat Milk       0.341715   8.838184   
21                                 Sourdough Bread       0.296534   1.773927   
53                                    Classic Soda       0.229671   4.016049   
25                                    Energy Drink       0.227197  12.784432   
98  Heavy Whipping Crean Ultra Pasteurized Grade A       0.165867   3.585926   
26                 Minis Original Saltine Crackers       0.163621   5.722764   
23                        Grade A Large White Eggs       0.156555   6.375359   
54            Vitamin Water Zero Squeezed Lemonade       0.142870   5.880439   
3                                 Shredded Carrots       0.119765   0.769275   
92                  Vitamin Water Zero Rise Orange       0.116245  10.511756   

    label  
24      0  
21      0  
53      0  
25      0  
98      0  
26      0

In [ ]:
# Sauvegarde des prédictions et validation (pour business_metrics.py)
np.save('val_proba.npy', val_proba)
np.save('y_val.npy', y_val.values if isinstance(y_val, pd.Series) else y_val)
X_val.to_pickle('X_val.pkl')

print("\nÉléments de validation sauvegardés avec succès :")
print("- val_proba.npy")
print("- y_val.npy")
print("- X_val.pkl")



Éléments de validation sauvegardés avec succès :
- val_proba.npy
- y_val.npy
- X_val.pkl
- best_threshold.npy
